# 1. Run identity and immutable configuration

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time
AUTHORIZE_REAL_INFERENCE = False
RUN_ID = "controlledrag-targeted-modern-judge-preregistered-v1"
print(RUN_ID, "AUTHORIZE_REAL_INFERENCE=", AUTHORIZE_REAL_INFERENCE)

# 2. Kaggle hardware validation

In [ ]:
BUNDLE_ROOT = Path.cwd().resolve()
if BUNDLE_ROOT.name == "notebooks":
    BUNDLE_ROOT = BUNDLE_ROOT.parent
sys.path.insert(0, str(BUNDLE_ROOT / "src"))
from validate_environment import inspect_environment
env = inspect_environment(require_t4x2=bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")))
print(json.dumps(env, indent=2, sort_keys=True))

# 3. Dependency installation

In [ ]:
INSTALL_PINNED_DEPENDENCIES = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if INSTALL_PINNED_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(BUNDLE_ROOT / "requirements/requirements-kaggle.txt")], check=True)
else:
    print("Local validation: dependency installation skipped; exact Kaggle lock remains frozen.")

# 4. Input ZIP discovery and checksum verification

In [ ]:
if not (BUNDLE_ROOT / "SHA256SUMS.txt").exists():
    candidates = list(Path("/kaggle/input").glob("**/CONTROLLEDRAG_TARGETED_JUDGE_INPUT/SHA256SUMS.txt")) if Path("/kaggle/input").exists() else []
    if len(candidates) != 1:
        raise RuntimeError("Could not uniquely discover the private input bundle")
    BUNDLE_ROOT = candidates[0].parent
from validate_input_bundle import validate_bundle
input_validation = validate_bundle(BUNDLE_ROOT)
print(json.dumps(input_validation, indent=2, sort_keys=True))
assert input_validation["ok"]

# 5. Frozen manifest summary

In [ ]:
summary=json.loads((BUNDLE_ROOT/"manifests/MANIFEST_SUMMARY.json").read_text())
print(json.dumps(summary,indent=2,sort_keys=True))
assert summary["main_eligible"]==600 and summary["primary_complete_pairs"]==194
assert summary["typical_eligible"]==83 and summary["disagreement_eligible"]==72

# 6. Model revision verification

In [ ]:
lock=json.loads((BUNDLE_ROOT/"MODEL_LOCK.json").read_text())
assert lock["model_id"]=="Qwen/Qwen2.5-7B-Instruct"
assert lock["revision"]=="a09a35458c702b33eeacc393d103063234e8bc28"
attached_model_path=os.environ.get("ATTACHED_MODEL_PATH","").strip()
if attached_model_path:
    attached_root=Path(attached_model_path)
    assert (attached_root/"MODEL_ID.txt").read_text().strip()==lock["model_id"]
    assert (attached_root/"REVISION.txt").read_text().strip()==lock["revision"]
    print("Attached snapshot identity sidecars verified; no weights loaded.")
elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    from huggingface_hub import model_info
    resolved=model_info(lock["model_id"],revision=lock["revision"],files_metadata=False).sha
    assert resolved==lock["revision"]
    print("Hugging Face metadata revision verified; no weights loaded.")
else:
    print(lock["model_id"], lock["revision"], "preparation metadata lock verified; no weights loaded")

# 7. Synthetic parser and schema tests

In [ ]:
test_env=dict(os.environ)
test_env["PYTHONPATH"]=str(BUNDLE_ROOT/"src")
tests=subprocess.run([sys.executable,"-m","unittest","discover","-s",str(BUNDLE_ROOT/"tests"),"-v"],cwd=BUNDLE_ROOT,env=test_env,text=True,capture_output=True)
print(tests.stdout); print(tests.stderr)
assert tests.returncode==0

# 8. Synthetic T4 memory probe

In [ ]:
probe_started=time.perf_counter()
if env.get("cuda_available") and env.get("gpu_count")==2:
    import torch
    for device in range(2):
        probe=torch.empty((4,512,1024),dtype=torch.float16,device=device)
        del probe
    torch.cuda.empty_cache()
    print("Synthetic allocation probe passed on two GPUs; no model loaded.")
else:
    print("Local validation: synthetic GPU allocation probe skipped.")
probe_seconds=max(time.perf_counter()-probe_started,1e-9)
synthetic_operations=2 if env.get("gpu_count")==2 else 1
print("Synthetic preparation throughput:",synthetic_operations/probe_seconds,"probe operations/second.")
print("Planning estimate only: approximately 45 minutes to 2 hours for real inference, excluding model download and Kaggle queue/startup; the synthetic allocation rate is not claimed as model throughput.")

# 9. Explicit human confirmation cell

In [ ]:
print("By changing only AUTHORIZE_REAL_INFERENCE to True, the operator confirms the frozen model, prompts, manifests, metrics, and thresholds.")
if not AUTHORIZE_REAL_INFERENCE:
    print("PREPARATION_VALIDATED_REAL_INFERENCE_NOT_AUTHORIZED")

# 10. Real dual-GPU inference

In [ ]:
OUTPUT_ROOT=Path("/kaggle/working/CONTROLLEDRAG_TARGETED_JUDGE_OUTPUT") if Path("/kaggle/working").exists() else BUNDLE_ROOT/"_synthetic_output"
if AUTHORIZE_REAL_INFERENCE:
    started=time.time()
    for subdir in ("raw_outputs","validation","analysis","tables","logs"):
        (OUTPUT_ROOT/subdir).mkdir(parents=True,exist_ok=True)
    (OUTPUT_ROOT/"ENVIRONMENT.json").write_text(json.dumps(env,indent=2,sort_keys=True)+"\n")
    (OUTPUT_ROOT/"MODEL_IDENTITY.json").write_text(json.dumps(lock,indent=2,sort_keys=True)+"\n")
    (OUTPUT_ROOT/"INPUT_CHECKSUM_VERIFICATION.json").write_text(json.dumps(input_validation,indent=2,sort_keys=True)+"\n")
    (OUTPUT_ROOT/"FINAL_RUN_CONFIG.yaml").write_text((BUNDLE_ROOT/"RUN_CONFIG.yaml").read_text())
    (OUTPUT_ROOT/"logs/warnings.log").write_text("")
    os.environ["AUTHORIZE_REAL_INFERENCE"]="1"
    cmd=["torchrun","--standalone","--nproc_per_node=2",str(BUNDLE_ROOT/"src/run_judge_distributed.py"),"--bundle-root",str(BUNDLE_ROOT),"--output-root",str(OUTPUT_ROOT),"--authorize-real-inference"]
    print(" ".join(cmd))
    with (OUTPUT_ROOT/"logs/torchrun.log").open("w") as log:
        subprocess.run(cmd,check=True,stdout=log,stderr=subprocess.STDOUT)
    (OUTPUT_ROOT/"logs/runtime.csv").write_text("stage,seconds\ninference,"+str(time.time()-started)+"\n")
else:
    print("Real inference skipped. No tokenizer or model was loaded.")

# 11. Shard merge

In [ ]:
if AUTHORIZE_REAL_INFERENCE:
    from merge_shards import merge
    print(merge(OUTPUT_ROOT))
else: print("Merge skipped until authorized inference creates two shards.")

# 12. Output validation

In [ ]:
if AUTHORIZE_REAL_INFERENCE:
    from validate_outputs import validate
    output_validation=validate(BUNDLE_ROOT,OUTPUT_ROOT)
    print(json.dumps(output_validation,indent=2))
    assert output_validation["ok"]
else: print("Output validation contract loaded; no real outputs exist.")

# 13. Preregistered analysis

In [ ]:
if AUTHORIZE_REAL_INFERENCE:
    from analyze_targeted_judge import analyze
    print(analyze(BUNDLE_ROOT,OUTPUT_ROOT))
else: print("Preregistered analysis skipped: no real outputs.")

# 14. Rebuttal table generation

In [ ]:
if AUTHORIZE_REAL_INFERENCE:
    from build_rebuttal_tables import build
    build(OUTPUT_ROOT)
else: print("Rebuttal table generation skipped: no real outputs.")

# 15. Output ZIP creation

In [ ]:
if AUTHORIZE_REAL_INFERENCE:
    from package_kaggle_outputs import package
    receipt={"status":"REAL_RUN_COMPLETE","run_id":RUN_ID,"model_id":lock["model_id"],"model_revision":lock["revision"],"started_unix":started,"completed_unix":time.time(),"authorization":True}
    (OUTPUT_ROOT/"RUN_RECEIPT.json").write_text(json.dumps(receipt,indent=2,sort_keys=True)+"\n")
    zip_path=OUTPUT_ROOT.parent/"CONTROLLEDRAG_TARGETED_JUDGE_OUTPUT.zip"
    print(package(OUTPUT_ROOT,zip_path))
else: print("Output ZIP packaging implementation passed synthetic unit validation.")

# 16. Final run receipt

In [ ]:
if AUTHORIZE_REAL_INFERENCE:
    print("REAL_RUN_COMPLETE_RETURN_OUTPUT_ZIP_FOR_PROMPT_B")
else:
    print("PREPARATION_VALIDATED_REAL_INFERENCE_NOT_AUTHORIZED")
    print("Estimated real inference runtime: approximately 45 minutes to 2 hours, excluding model download and Kaggle queue/startup.")